# WikiArt Style Classification — Test 12: DINOv2 ViT-Large at 448px

*ViT-Large DINOv2 (307M params) at 448px with LLRD, lighter augmentation, and multi-scale TTA.*

**Key changes vs Test 11 (73.16% test):**
- ViT-Base -> ViT-Large (86M params — biggest lever available)
- Resolution 336 -> 448px (1024 patches, richer fine-grained detail)
- LLRD decay 0.80 -> 0.85
- Augmentation dialled back (mixup 0.65->0.50, cutmix 1.0->0.80, erasing 0.22->0.20)
- Label smoothing 0.08 -> 0.06
- 65 fine-tune epochs
- Multi-scale TTA at final evaluation (0.9x, 1.0x, 1.1x + flip = 6 passes)

In [1]:
# ─── Installs (skip if already installed) ───────────────────────────────────
# !pip install timm>=1.0.0 torchvision pandas pillow tqdm
import importlib
for pkg in ['timm', 'torchvision', 'pandas', 'PIL', 'tqdm']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — please install')

  ✓ timm
  ✓ torchvision
  ✓ pandas
  ✓ PIL
  ✓ tqdm


In [2]:
import os, sys, json, time, random, math, warnings
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass
from typing import Tuple, List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from timm.utils import ModelEmaV2
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy

warnings.filterwarnings('ignore', category=UserWarning)

print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch : 2.10.0+cu128
timm    : 1.0.25
CUDA    : True
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM    : 8.6 GB


In [3]:
# --- Configuration ----------------------------------------------------------
@dataclass
class TrainConfig:
    # DINOv2 ViT-Base/14 with register tokens (~86M params)
    model_name: str = 'vit_base_patch14_reg4_dinov2.lvd142m'
    # 448 = 32 x 14  ->  32x32 = 1024 patches  (interpolated from native 518px)
    image_size: int = 448

    batch_size: int = 8
    effective_batch_size: int = 64   # grad accumulation = 64 // 8 = 8 steps
    num_workers: int = 0
    use_weighted_sampler: bool = False

    head_only_epochs: int = 5
    finetune_epochs: int = 80

    head_lr: float = 5e-4

    finetune_head_lr: float = 3e-5
    llrd_decay: float = 0.80

    warmup_epochs: int = 5

    weight_decay: float = 0.03
    label_smoothing: float = 0.04
    mixup_alpha: float = 0.50
    mixup_alpha_final: float = 0.00
    cutmix_alpha: float = 0.80
    cutmix_alpha_final: float = 0.00
    regularization_cooldown_start: float = 0.50
    hard_label_finetune_start: float = 0.80
    randaug_num_ops: int = 2
    randaug_magnitude: int = 11
    random_erasing_prob: float = 0.20
    ema_decay: float = 0.9999
    grad_clip: float = 1.0

    seed: int = 42

    output_dir: str = 'models'
    warmstart_checkpoint: str = 'models/wikiart_test11_dinov2_llrd_best.pt'
    model_save_name: str = 'wikiart_test12_vitlarge_448_best.pt'
    history_csv: str = 'models/results/wikiart_test12_vitlarge_448_history.csv'
    summary_csv: str = 'models/results/wikiart_tests_1_to_12_summary.csv'

    @property
    def accumulate_steps(self) -> int:
        return max(1, self.effective_batch_size // self.batch_size)

    @property
    def total_epochs(self) -> int:
        return self.head_only_epochs + self.finetune_epochs


cfg = TrainConfig()
print(cfg)
print(f'Gradient accumulation steps: {cfg.accumulate_steps}')
print(f'Total epochs: {cfg.total_epochs}')

TrainConfig(model_name='vit_base_patch14_reg4_dinov2.lvd142m', image_size=448, batch_size=8, effective_batch_size=64, num_workers=0, use_weighted_sampler=False, head_only_epochs=5, finetune_epochs=80, head_lr=0.0005, finetune_head_lr=3e-05, llrd_decay=0.8, warmup_epochs=5, weight_decay=0.03, label_smoothing=0.04, mixup_alpha=0.5, mixup_alpha_final=0.0, cutmix_alpha=0.8, cutmix_alpha_final=0.0, regularization_cooldown_start=0.5, hard_label_finetune_start=0.8, randaug_num_ops=2, randaug_magnitude=11, random_erasing_prob=0.2, ema_decay=0.9999, grad_clip=1.0, seed=42, output_dir='models', warmstart_checkpoint='models/wikiart_test11_dinov2_llrd_best.pt', model_save_name='wikiart_test12_vitlarge_448_best.pt', history_csv='models/results/wikiart_test12_vitlarge_448_history.csv', summary_csv='models/results/wikiart_tests_1_to_12_summary.csv')
Gradient accumulation steps: 8
Total epochs: 85


In [4]:
# ─── Reproducibility & Device ────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [5]:
# ─── Data Paths ──────────────────────────────────────────────────────────────
def discover_project_root() -> Path:
    root = Path('.').resolve()
    for _ in range(6):
        if (root / 'datasets').exists():
            return root
        root = root.parent
    raise FileNotFoundError('Could not find project root (no datasets/ dir found)')


def discover_paths(project_root: Path) -> Tuple[Path, Path, Path]:
    wikiart_dir = project_root / 'datasets' / 'Wikiart'
    train_csv = wikiart_dir / 'style_train.csv'
    val_csv   = wikiart_dir / 'style_val.csv'
    if not train_csv.exists() or not val_csv.exists():
        raise FileNotFoundError(
            f'Missing style_train.csv or style_val.csv in {wikiart_dir}')
    return wikiart_dir, train_csv, val_csv


def load_style_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=['relative_path', 'label'])
    df['label'] = df['label'].astype(int)
    return df


def filter_existing_rows(df: pd.DataFrame, image_root: Path,
                          split_name: str = '') -> pd.DataFrame:
    mask = df['relative_path'].apply(lambda p: (image_root / p).exists())
    n_missing = (~mask).sum()
    if n_missing > 0:
        print(f'  [{split_name}] Skipping {n_missing} missing files')
    return df[mask].reset_index(drop=True)


def make_eval_split(val_df: pd.DataFrame,
                    seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Stratified 50/50 split of val set into val and test."""
    sampled_parts = []
    for _, grp in val_df.groupby('label', sort=False):
        if len(grp) > 1:
            sampled_parts.append(grp.sample(frac=0.5, random_state=seed))
        else:
            sampled_parts.append(grp)
    eval_test = pd.concat(sampled_parts, axis=0)
    eval_val  = val_df.drop(index=eval_test.index)
    if len(eval_val) == 0:
        eval_val  = val_df.sample(frac=0.5, random_state=seed)
        eval_test = val_df.drop(index=eval_val.index)
    cols = ['relative_path', 'label']
    return eval_val[cols].reset_index(drop=True), eval_test[cols].reset_index(drop=True)


# ── Locate data ──────────────────────────────────────────────────────────────
project_root = discover_project_root()
wikiart_dir, train_csv_path, val_csv_path = discover_paths(project_root)
print(f'Project root : {project_root}')
print(f'WikiArt dir  : {wikiart_dir}')

train_df_raw = load_style_csv(train_csv_path)
val_df_raw   = load_style_csv(val_csv_path)

train_df    = filter_existing_rows(train_df_raw, wikiart_dir, 'train')
val_all_df  = filter_existing_rows(val_df_raw,   wikiart_dir, 'val')
eval_val_df, eval_test_df = make_eval_split(val_all_df, seed=cfg.seed)

num_classes = int(train_df['label'].max() + 1)

print(f'\nClasses : {num_classes}')
print(f'Train   : {len(train_df):,}')
print(f'Val     : {len(eval_val_df):,}')
print(f'Test    : {len(eval_test_df):,}')

counts = train_df['label'].value_counts().sort_index()
print(f'\nClass distribution — min: {counts.min()}, max: {counts.max()}, '
      f'std: {counts.std():.0f}')

Project root : C:\Users\Thijs\Desktop\Ai Art Critic
WikiArt dir  : C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Wikiart
  [train] Skipping 2 missing files

Classes : 27
Train   : 57,023
Val     : 12,213
Test    : 12,208

Class distribution — min: 69, max: 9142, std: 2288


In [6]:
# ─── Dataset ─────────────────────────────────────────────────────────────────
class WikiArtStyleDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, image_root: Path, transform=None):
        self.rows       = rows.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform  = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int):
        row   = self.rows.iloc[idx]
        img   = Image.open(self.image_root / row['relative_path']).convert('RGB')
        label = int(row['label'])
        if self.transform is not None:
            img = self.transform(img)
        return img, label

In [7]:
# ─── Transforms ──────────────────────────────────────────────────────────────
# DINOv2 uses standard ImageNet normalization (verified via timm data config)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


def build_transforms(image_size: int, magnitude: int = 11,
                     erasing_prob: float = 0.22):
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.55, 1.0),
                                     ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=magnitude),
        transforms.ColorJitter(brightness=0.15, contrast=0.15,
                                saturation=0.15, hue=0.04),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
        transforms.RandomErasing(p=erasing_prob, scale=(0.02, 0.18),
                                  ratio=(0.3, 3.3), value='random'),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    return train_tfms, eval_tfms


train_tfms, eval_tfms = build_transforms(
    cfg.image_size,
    magnitude=cfg.randaug_magnitude,
    erasing_prob=cfg.random_erasing_prob,
)
print('Transforms built.')
print(f'  Train: {train_tfms}')
print(f'  Eval : {eval_tfms}')

Transforms built.
  Train: Compose(
    RandomResizedCrop(size=(448, 448), scale=(0.55, 1.0), ratio=(0.75, 1.33), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandAugment(num_ops=2, magnitude=11, num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
    ColorJitter(brightness=(0.85, 1.15), contrast=(0.85, 1.15), saturation=(0.85, 1.15), hue=(-0.04, 0.04))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    RandomErasing(p=0.2, scale=(0.02, 0.18), ratio=(0.3, 3.3), value=random, inplace=False)
)
  Eval : Compose(
    Resize(size=510, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(448, 448))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [8]:
# ─── DataLoaders ─────────────────────────────────────────────────────────────
def create_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    class_counts   = np.bincount(labels, minlength=int(labels.max()) + 1)
    class_weights  = 1.0 / np.maximum(class_counts.astype(float), 1.0)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(labels),
        replacement = True,
    )


def pick_num_workers(requested: int) -> int:
    if requested < 0:
        return max(0, (os.cpu_count() or 1) - 1)
    return requested


num_workers = pick_num_workers(cfg.num_workers)
pin_mem     = device.type == 'cuda'

train_ds = WikiArtStyleDataset(train_df,    wikiart_dir, train_tfms)
val_ds   = WikiArtStyleDataset(eval_val_df, wikiart_dir, eval_tfms)
test_ds  = WikiArtStyleDataset(eval_test_df,wikiart_dir, eval_tfms)

sampler = (create_weighted_sampler(train_df['label'].values)
           if cfg.use_weighted_sampler else None)

loader_kw = dict(num_workers=num_workers, pin_memory=pin_mem,
                 persistent_workers=(num_workers > 0))

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                          sampler=sampler, shuffle=(sampler is None),
                          drop_last=True, **loader_kw)
val_loader   = DataLoader(val_ds,  batch_size=cfg.batch_size,
                          shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size,
                          shuffle=False, **loader_kw)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

Train batches : 7127
Val batches   : 1527
Test batches  : 1526


In [9]:
# ─── MixUp / CutMix ──────────────────────────────────────────────────────────
# timm's Mixup handles both MixUp and CutMix with soft targets + label smoothing
mixup_fn = Mixup(
    mixup_alpha   = cfg.mixup_alpha,
    cutmix_alpha  = cfg.cutmix_alpha,
    cutmix_minmax = None,
    prob          = 1.0,    # always apply one of MixUp or CutMix during training
    switch_prob   = 0.5,    # 50% chance to use CutMix vs MixUp
    mode          = 'batch',
    label_smoothing = cfg.label_smoothing,
    num_classes   = num_classes,
)

# Loss: soft-target CE for training (mixup produces soft labels)
#        standard CE for evaluation
train_criterion = SoftTargetCrossEntropy()
hard_train_criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
eval_criterion  = nn.CrossEntropyLoss()
print('MixUp/CutMix and loss functions ready.')

MixUp/CutMix and loss functions ready.


In [10]:
# --- Model ------------------------------------------------------------------
print(f'Creating model: {cfg.model_name} at {cfg.image_size}px')
model = timm.create_model(
    cfg.model_name,
    pretrained  = True,
    num_classes = num_classes,
    img_size    = cfg.image_size,   # timm interpolates pos_embed from native 518px
)
warmstart_path = Path(cfg.warmstart_checkpoint)
if warmstart_path.exists():
    print(f'Loading warm-start checkpoint: {warmstart_path}')
    ckpt = torch.load(warmstart_path, map_location='cpu')
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        ckpt = ckpt['state_dict']
    if isinstance(ckpt, dict):
        ckpt = {k.replace('module.', ''): v for k, v in ckpt.items()}
        model_state = model.state_dict()
        filtered_ckpt = {}
        skipped_shape = []
        for k, v in ckpt.items():
            if k in model_state and model_state[k].shape == v.shape:
                filtered_ckpt[k] = v
            elif k in model_state:
                skipped_shape.append((k, tuple(v.shape), tuple(model_state[k].shape)))
        missing, unexpected = model.load_state_dict(filtered_ckpt, strict=False)
        print(f'  Warm-start loaded. Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}')
        if skipped_shape:
            print(f'  Skipped {len(skipped_shape)} shape-mismatched tensors (example: {skipped_shape[0][0]})')
    else:
        print('Warm-start checkpoint format not recognized; skipping warm-start load.')
else:
    print(f'Warm-start checkpoint not found: {warmstart_path} (using ImageNet-pretrained init only)')
model = model.to(device)

data_cfg = timm.data.resolve_model_data_config(model)
print(f'Model data config:')
print(f'  input_size  : {data_cfg["input_size"]}')
print(f'  mean / std  : {data_cfg["mean"]} / {data_cfg["std"]}')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable_params/1e6:.1f}M')

n_blocks = len(model.blocks)
print(f'Transformer blocks: {n_blocks}')

with torch.no_grad():
    dummy = torch.randn(2, 3, cfg.image_size, cfg.image_size, device=device)
    out   = model(dummy)
    print(f'Output shape: {out.shape}  (expected: [2, {num_classes}])')

Creating model: vit_base_patch14_reg4_dinov2.lvd142m at 448px
Loading warm-start checkpoint: models\wikiart_test11_dinov2_llrd_best.pt
  Warm-start loaded. Missing keys: 1 | Unexpected keys: 0
  Skipped 1 shape-mismatched tensors (example: pos_embed)
Model data config:
  input_size  : (3, 518, 518)
  mean / std  : (0.485, 0.456, 0.406) / (0.229, 0.224, 0.225)
Total params    : 86.3M
Trainable params: 86.3M
Transformer blocks: 12
Output shape: torch.Size([2, 27])  (expected: [2, 27])


In [11]:
# ─── EMA ─────────────────────────────────────────────────────────────────────
ema_model = ModelEmaV2(model, decay=cfg.ema_decay, device=device)
print(f'EMA model created (decay={cfg.ema_decay})')

EMA model created (decay=0.9999)


In [12]:
# ─── LLRD Optimizer Builder ───────────────────────────────────────────────────
def build_llrd_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.Optimizer:
    """
    Build AdamW with Layer-wise Learning Rate Decay (LLRD).

    Layer LR assignment (from lowest to highest LR):
      Stem (patch_embed, pos_embed, cls/reg tokens)
        → Block 0 (earliest)
        → Block 1
        → ...
        → Block N-1 (latest)
        → Final norm + Head   ← finetune_head_lr (highest)

    Formula: LR_block_i = finetune_head_lr × decay^(depth_from_head)
    where depth_from_head = 1 for the last block, N for the first block.
    """
    head_lr = cfg.finetune_head_lr
    decay   = cfg.llrd_decay
    wd      = cfg.weight_decay

    blocks  = list(model.blocks)
    n_blocks = len(blocks)

    # Parameters that should NOT have weight decay
    NO_WD_PATTERNS = {'bias', '.norm', 'norm', 'layer_norm', 'pos_embed',
                      'cls_token', 'reg_token', 'dist_token'}

    def is_no_wd(name: str) -> bool:
        return any(p in name for p in NO_WD_PATTERNS)

    param_groups  = []
    assigned_ids  = set()

    def add_group(named_params: list, lr: float):
        wd_params  = []
        no_wd_params = []
        for name, p in named_params:
            if not p.requires_grad or id(p) in assigned_ids:
                continue
            assigned_ids.add(id(p))
            if is_no_wd(name):
                no_wd_params.append(p)
            else:
                wd_params.append(p)
        if wd_params:
            param_groups.append({'params': wd_params,    'lr': lr, 'weight_decay': wd})
        if no_wd_params:
            param_groups.append({'params': no_wd_params, 'lr': lr, 'weight_decay': 0.0})

    # ── 1. Head + final norm  (full LR, depth = 0) ───────────────────────────
    head_named = [(n, p) for n, p in model.named_parameters()
                  if n.startswith('head') or n.startswith('norm.') or n.startswith('fc_norm')]
    add_group(head_named, head_lr)

    # ── 2. Transformer blocks (from last → first, decreasing LR) ─────────────
    for depth, block_idx in enumerate(range(n_blocks - 1, -1, -1)):
        block = blocks[block_idx]
        block_lr = head_lr * (decay ** (depth + 1))
        block_named = [(f'blocks.{block_idx}.{n}', p) for n, p in block.named_parameters()]
        add_group(block_named, block_lr)

    # ── 3. Stem: patch_embed, pos_embed, cls_token, reg_tokens ───────────────
    stem_lr      = head_lr * (decay ** (n_blocks + 1))
    stem_named   = [(n, p) for n, p in model.named_parameters()
                    if p.requires_grad and id(p) not in assigned_ids]
    add_group(stem_named, stem_lr)

    # ── Print LLRD summary ───────────────────────────────────────────────────
    print(f'\nLLRD Summary (n_blocks={n_blocks}, decay={decay}, head_lr={head_lr:.1e})')
    print(f'  Stem   (block -∞): {stem_lr:.2e}')
    print(f'  Block   0 (first): {head_lr * decay**n_blocks:.2e}')
    print(f'  Block   {n_blocks//2} (mid)  : {head_lr * decay**(n_blocks//2+1):.2e}')
    print(f'  Block {n_blocks-1:2d} (last) : {head_lr * decay**1:.2e}')
    print(f'  Head             : {head_lr:.2e}')
    print(f'  Param groups     : {len(param_groups)}')

    return torch.optim.AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)


print('LLRD optimizer builder defined.')

LLRD optimizer builder defined.


In [13]:
# ─── LR Scheduler Builder ────────────────────────────────────────────────────
def build_scheduler(optimizer: torch.optim.Optimizer,
                    warmup_steps: int, total_steps: int):
    """Linear warmup → cosine annealing to 0."""
    warmup_sched  = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0,
        total_iters=warmup_steps,
    )
    cosine_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps - warmup_steps, eta_min=0.0,
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[warmup_steps],
    )


print('Scheduler builder defined.')

Scheduler builder defined.


In [14]:
# --- Accuracy Helpers -------------------------------------------------------
import torch.nn.functional as F

@torch.no_grad()
def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> float:
    _, pred = logits.topk(k, dim=1, largest=True, sorted=True)
    correct = pred.eq(targets.view(-1, 1).expand_as(pred))
    return correct[:, :k].reshape(-1).float().sum().item() / targets.size(0)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader,
             criterion: nn.Module, device: torch.device,
             tta_flip: bool = False,
             tta_scales: list = None) -> Tuple[float, float, float]:
    """Returns (loss, top1_acc, top5_acc).
    tta_scales: extra scale factors e.g. [0.9, 1.1] applied on top of flip TTA.
    """
    model.eval()
    total_loss, total_top1, total_top5, n = 0.0, 0.0, 0.0, 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        bs = labels.size(0)
        h, w = images.shape[-2:]

        aug_views = [images]
        if tta_flip:
            aug_views.append(images.flip(-1))
        if tta_scales:
            for s in tta_scales:
                new_h, new_w = int(h * s), int(w * s)
                scaled = F.interpolate(images, size=(new_h, new_w),
                                       mode='bicubic', align_corners=False)
                y0 = max(0, (new_h - h) // 2)
                x0 = max(0, (new_w - w) // 2)
                scaled = scaled[:, :, y0:y0+min(new_h, h), x0:x0+min(new_w, w)]
                if scaled.shape[-2:] != (h, w):
                    scaled = F.interpolate(scaled, size=(h, w),
                                           mode='bicubic', align_corners=False)
                aug_views.append(scaled)
                if tta_flip:
                    aug_views.append(scaled.flip(-1))

        logits = sum(model(v) for v in aug_views) / len(aug_views)

        loss = criterion(logits, labels)
        total_loss += loss.item() * bs
        total_top1 += topk_accuracy(logits, labels, k=1) * bs
        total_top5 += topk_accuracy(logits, labels, k=min(5, num_classes)) * bs
        n += bs

    return total_loss / n, total_top1 / n, total_top5 / n


print('Evaluation helpers defined.')

Evaluation helpers defined.


In [15]:
# --- Phase 1: Head-only training --------------------------------------------
print('=' * 70)
print('PHASE 1 -- Head-only training')
print('=' * 70)

for name, param in model.named_parameters():
    param.requires_grad = name.startswith('head')

head_params = [p for p in model.parameters() if p.requires_grad]
optimizer_head = torch.optim.AdamW(
    head_params,
    lr=cfg.head_lr,
    weight_decay=cfg.weight_decay,
)

history = []

for epoch in range(1, cfg.head_only_epochs + 1):
    model.train()
    optimizer_head.zero_grad()
    running_loss, running_correct, n_seen = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.head_only_epochs} [head]',
                leave=False)

    for step, (images, labels) in enumerate(pbar, 1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        images_mix, labels_mix = mixup_fn(images, labels)

        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images_mix)
            loss   = train_criterion(logits, labels_mix) / cfg.accumulate_steps

        loss.backward()

        if step % cfg.accumulate_steps == 0 or step == len(train_loader):
            nn.utils.clip_grad_norm_(head_params, cfg.grad_clip)
            optimizer_head.step()
            optimizer_head.zero_grad()
            ema_model.update(model)

        with torch.no_grad():
            running_correct += logits.argmax(1).eq(labels).sum().item()
            n_seen += labels.size(0)
        running_loss += loss.item() * cfg.accumulate_steps
        pbar.set_postfix(loss=f'{running_loss/step:.3f}',
                         acc=f'{running_correct/n_seen:.3f}')

    val_loss, val_top1, val_top5 = evaluate(
        ema_model.module, val_loader, eval_criterion, device, tta_flip=False)

    lr_now = optimizer_head.param_groups[0]['lr']
    print(f'  Epoch {epoch:3d}  '
          f'lr={lr_now:.2e}  '
          f'val_loss={val_loss:.4f}  '
          f'val_top1={val_top1:.4f}  '
          f'val_top5={val_top5:.4f}')

    history.append({
        'epoch': epoch, 'stage': 'head-only',
        'lr': lr_now,
        'train_loss': running_loss / len(train_loader),
        'train_acc': running_correct / n_seen,
        'val_loss': val_loss, 'val_top1': val_top1, 'val_top5': val_top5,
    })

print('Phase 1 complete.')

PHASE 1 -- Head-only training


Epoch 1/5 [head]:   0%|          | 0/7127 [00:00<?, ?it/s]

c:\Users\Thijs\Desktop\Ai Art Critic\.venv\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
c:\Users\Thijs\Desktop\Ai Art Critic\.venv\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (107327830 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  Epoch   1  lr=5.00e-04  val_loss=0.8377  val_top1=0.7449  val_top5=0.9779


Epoch 2/5 [head]:   0%|          | 0/7127 [00:00<?, ?it/s]

...

  Epoch   3  lr=5.00e-04  val_loss=0.8253  val_top1=0.7453  val_top5=0.9784


Epoch 4/5 [head]:   0%|          | 0/7127 [00:00<?, ?it/s]

  Epoch   4  lr=5.00e-04  val_loss=0.8205  val_top1=0.7462  val_top5=0.9781


Epoch 5/5 [head]:   0%|          | 0/7127 [00:00<?, ?it/s]

  Epoch   5  lr=5.00e-04  val_loss=0.8165  val_top1=0.7464  val_top5=0.9783
Phase 1 complete.


In [16]:
# ─── Phase 2: Full fine-tune with LLRD ───────────────────────────────────────
print('=' * 70)
print('PHASE 2 — Full fine-tune with LLRD')
print('=' * 70)

# Unfreeze all parameters
for param in model.parameters():
    param.requires_grad = True

optimizer = build_llrd_optimizer(model, cfg)

warmup_steps = cfg.warmup_epochs * len(train_loader) // cfg.accumulate_steps
total_steps  = cfg.finetune_epochs * len(train_loader) // cfg.accumulate_steps
scheduler    = build_scheduler(optimizer, warmup_steps, total_steps)

print(f'\nWarmup steps: {warmup_steps}  |  Total steps: {total_steps}')

# Mixed precision scaler
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

best_val_top1  = 0.0
best_epoch     = 0
best_ckpt_path = Path(cfg.output_dir) / cfg.model_save_name
best_ckpt_path.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()

for epoch in range(1, cfg.finetune_epochs + 1):
    model.train()
    optimizer.zero_grad()
    running_loss, running_correct, n_seen = 0.0, 0, 0
    global_epoch = cfg.head_only_epochs + epoch

    progress = (epoch - 1) / max(1, cfg.finetune_epochs - 1)
    if progress >= cfg.regularization_cooldown_start:
        t = (progress - cfg.regularization_cooldown_start) / max(1e-8, (1.0 - cfg.regularization_cooldown_start))
    else:
        t = 0.0
    epoch_mixup = cfg.mixup_alpha + (cfg.mixup_alpha_final - cfg.mixup_alpha) * t
    epoch_cutmix = cfg.cutmix_alpha + (cfg.cutmix_alpha_final - cfg.cutmix_alpha) * t
    use_hard_targets = progress >= cfg.hard_label_finetune_start
    epoch_mixup_fn = Mixup(
        mixup_alpha=epoch_mixup,
        cutmix_alpha=epoch_cutmix,
        cutmix_minmax=None,
        prob=1.0,
        switch_prob=0.5,
        mode='batch',
        label_smoothing=cfg.label_smoothing,
        num_classes=num_classes,
    )

    pbar = tqdm(train_loader,
                desc=f'Epoch {global_epoch}/{cfg.total_epochs} [ft]',
                leave=False)

    for step, (images, labels) in enumerate(pbar, 1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if use_hard_targets:
            images_mix, labels_mix = images, labels
        else:
            images_mix, labels_mix = epoch_mixup_fn(images, labels)

        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images_mix)
            if use_hard_targets:
                loss = hard_train_criterion(logits, labels_mix) / cfg.accumulate_steps
            else:
                loss = train_criterion(logits, labels_mix) / cfg.accumulate_steps

        scaler.scale(loss).backward()

        if step % cfg.accumulate_steps == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            ema_model.update(model)

        with torch.no_grad():
            running_correct += logits.argmax(1).eq(labels).sum().item()
            n_seen += labels.size(0)
        running_loss += loss.item() * cfg.accumulate_steps

        if step % max(1, len(train_loader) // 4) == 0:
            pbar.set_postfix(loss=f'{running_loss/step:.3f}',
                             acc=f'{running_correct/n_seen:.3f}')

    # ── Validate with EMA model ───────────────────────────────────────────────
    val_loss, val_top1, val_top5 = evaluate(
        ema_model.module, val_loader, eval_criterion, device, tta_flip=True)

    lr_now = optimizer.param_groups[0]['lr']  # head LR
    elapsed = (time.time() - t0) / 60

    marker = ''
    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        best_epoch    = global_epoch
        torch.save(ema_model.module.state_dict(), best_ckpt_path)
        marker = '  ← best'

    print(f'  Epoch {global_epoch:3d}  '
          f'lr={lr_now:.2e}  '
          f'mix={epoch_mixup:.2f}/{epoch_cutmix:.2f}  '
          f'mode={"hard" if use_hard_targets else "mix"}  '
          f'val_loss={val_loss:.4f}  '
          f'val_top1={val_top1:.4f}  '
          f'best={best_val_top1:.4f}'
          f'{marker}'
          f'  [{elapsed:.1f}m]')

    history.append({
        'epoch': global_epoch, 'stage': 'fine-tune',
        'lr': lr_now,
        'train_loss': running_loss / len(train_loader),
        'train_acc': running_correct / n_seen,
        'val_loss': val_loss, 'val_top1': val_top1, 'val_top5': val_top5,
    })

print(f'\nPhase 2 complete. Best val top-1: {best_val_top1:.4f} @ epoch {best_epoch}')

PHASE 2 — Full fine-tune with LLRD

LLRD Summary (n_blocks=12, decay=0.8, head_lr=3.0e-05)
  Stem   (block -∞): 1.65e-06
  Block   0 (first): 2.06e-06
  Block   6 (mid)  : 6.29e-06
  Block 11 (last) : 2.40e-05
  Head             : 3.00e-05
  Param groups     : 28

Warmup steps: 4454  |  Total steps: 71270


Epoch 6/85 [ft]:   0%|          | 0/7127 [00:00<?, ?it/s]

  Epoch   6  lr=6.24e-06  mix=0.50/0.80  mode=mix  val_loss=0.8089  val_top1=0.7488  best=0.7488  ← best  [89.3m]


Epoch 7/85 [ft]:   0%|          | 0/7127 [00:00<?, ?it/s]

  Epoch   7  lr=1.22e-05  mix=0.50/0.80  mode=mix  val_loss=0.8040  val_top1=0.7501  best=0.7501  ← best  [178.3m]


...

  Epoch  83  lr=5.20e-08  mix=0.03/0.04  mode=hard  val_loss=0.8308  val_top1=0.7825  best=0.7831  [6935.5m]


Epoch 84/85 [ft]:   0%|          | 0/7127 [00:00<?, ?it/s]

  Epoch  84  lr=1.29e-08  mix=0.01/0.02  mode=hard  val_loss=0.8332  val_top1=0.7824  best=0.7831  [7031.6m]


Epoch 85/85 [ft]:   0%|          | 0/7127 [00:00<?, ?it/s]

  Epoch  85  lr=1.66e-12  mix=0.00/0.00  mode=hard  val_loss=0.8355  val_top1=0.7825  best=0.7831  [7114.4m]

Phase 2 complete. Best val top-1: 0.7831 @ epoch 72


In [17]:
# ─── Save Training History ────────────────────────────────────────────────────
history_df = pd.DataFrame(history)
Path(cfg.history_csv).parent.mkdir(parents=True, exist_ok=True)
history_df.to_csv(cfg.history_csv, index=False)
print(f'History saved → {cfg.history_csv}')
print(history_df.tail(10).to_string())

History saved → models/results/wikiart_test12_vitlarge_448_history.csv
    epoch      stage            lr  train_loss  train_acc  val_loss  val_top1  val_top5
75     76  fine-tune  1.051038e-06    0.192973   0.981005  0.807992  0.781872  0.965774
76     77  fine-tune  8.322590e-07    0.191740   0.981111  0.811927  0.781790  0.965610
77     78  fine-tune  6.383416e-07    0.191950   0.982321  0.815747  0.781872  0.965610
78     79  fine-tune  4.696263e-07    0.191549   0.981619  0.819273  0.782281  0.965283
79     80  fine-tune  3.264090e-07    0.191044   0.981865  0.822443  0.782691  0.964955
80     81  fine-tune  2.089412e-07    0.190681   0.981882  0.825422  0.782854  0.964464
81     82  fine-tune  1.174289e-07    0.189993   0.982689  0.828179  0.782445  0.964218
82     83  fine-tune  5.203269e-08    0.190359   0.982935  0.830807  0.782527  0.964300
83     84  fine-tune  1.286737e-08    0.189502   0.982864  0.833237  0.782445  0.963727
84     85  fine-tune  1.658059e-12    0.190531   

In [18]:
# --- Final Evaluation on Test Set -------------------------------------------
print('Loading best checkpoint...')
best_model = timm.create_model(
    cfg.model_name, pretrained=False, num_classes=num_classes,
    img_size=cfg.image_size)
best_model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
best_model = best_model.to(device)

# Val: flip TTA only
val_loss, val_top1, val_top5 = evaluate(
    best_model, val_loader, eval_criterion, device, tta_flip=True)
print(f'Val  (flip TTA)        -- loss={val_loss:.4f}  top1={val_top1:.4f}  top5={val_top5:.4f}')

# Test: multi-scale TTA (0.9x, 1.0x, 1.1x) + flip = 6 passes
test_loss, test_top1, test_top5 = evaluate(
    best_model, test_loader, eval_criterion, device,
    tta_flip=True, tta_scales=[0.85, 0.925, 1.075, 1.15])
print(f'Test (multi-scale TTA) -- loss={test_loss:.4f}  top1={test_top1:.4f}  top5={test_top5:.4f}')

print(f'>>> Test 12 style accuracy: {test_top1*100:.2f}% <<<  (target: >80%, test11: 73.16%)')

Loading best checkpoint...
Val  (flip TTA)        -- loss=0.7911  top1=0.7831  top5=0.9675
Test (multi-scale TTA) -- loss=0.8336  top1=0.7733  top5=0.9649
>>> Test 12 style accuracy: 77.33% <<<  (target: >80%, test11: 73.16%)


In [19]:
# ─── Per-class Accuracy Analysis ─────────────────────────────────────────────
@torch.no_grad()
def per_class_accuracy(model: nn.Module, loader: DataLoader,
                        n_classes: int, device: torch.device,
                        tta_flip: bool = True) -> pd.DataFrame:
    model.eval()
    correct = np.zeros(n_classes)
    total   = np.zeros(n_classes)
    for images, labels in tqdm(loader, desc='Per-class eval', leave=False):
        images = images.to(device, non_blocking=True)
        labels_np = labels.numpy()
        logits = model(images)
        if tta_flip:
            logits = (logits + model(images.flip(-1))) * 0.5
        preds = logits.argmax(1).cpu().numpy()
        for c in range(n_classes):
            mask = labels_np == c
            total[c]   += mask.sum()
            correct[c] += (preds[mask] == c).sum()

    acc = np.where(total > 0, correct / total, 0.0)
    df  = pd.DataFrame({'class_id': np.arange(n_classes),
                        'n_total':  total.astype(int),
                        'n_correct': correct.astype(int),
                        'accuracy': acc})
    return df.sort_values('accuracy')


pc_df = per_class_accuracy(best_model, test_loader, num_classes, device)
print('\nPer-class accuracy (worst → best):')
print(pc_df.to_string(index=False))

Per-class eval:   0%|          | 0/1526 [00:00<?, ?it/s]


Per-class accuracy (worst → best):
 class_id  n_total  n_correct  accuracy
       10      140         59  0.421429
       16       47         21  0.446809
       25       32         16  0.500000
       20      968        592  0.611570
        6       72         46  0.638889
        1       14          9  0.642857
        2       16         11  0.687500
        9     1010        707  0.700000
        7      335        236  0.704478
       24      679        490  0.721649
        3      650        478  0.735385
       19      222        166  0.747748
       14      200        155  0.775000
       11      201        156  0.776119
        0      417        326  0.781775
       23     1052        828  0.787072
       21     1610       1273  0.790683
       13      192        154  0.802083
       15      360        290  0.805556
        8      208        171  0.822115
        5      242        203  0.838843
       18       76         64  0.842105
       12     1959       1654  0.844308
    

In [20]:
# ─── Update Summary CSV ───────────────────────────────────────────────────────
import datetime

notebook_name = 'wikiart_style_classification_test12_dinov2_448_warmstart.ipynb'

new_row = {
    'notebook':       notebook_name,
    'exists':         True,
    'model_name':     cfg.model_name,
    'val_loss':       val_loss,
    'val_top1':       val_top1,
    'val_top5':       val_top5,
    'test_loss':      test_loss,
    'test_top1':      test_top1,
    'test_top5':      test_top5,
    'best_val_top1':  best_val_top1,
    'best_epoch':     float(best_epoch),
    'experiment':     'test12',
    'saved_at_utc':   datetime.datetime.utcnow().isoformat(),
    'val_style_acc':  val_top1,
    'val_artist_acc': '',
    'val_emotion_acc':'',
    'test_style_acc': test_top1,
    'test_artist_acc':'',
    'test_emotion_acc':'',
    'best_combined_score': '',
}

# Load existing summary (test 1-9)
prev_summary_csv = Path('models/results/wikiart_tests_1_to_11_summary.csv')
if prev_summary_csv.exists():
    summary_df = pd.read_csv(prev_summary_csv)
else:
    summary_df = pd.DataFrame()

new_row_df = pd.DataFrame([new_row])
summary_df = pd.concat([summary_df, new_row_df], ignore_index=True)

summary_path = Path(cfg.summary_csv)
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print(f'Summary saved → {summary_path}')

print('\n=== FINAL RESULTS ===')
print(f'  Model       : {cfg.model_name}')
print(f'  Best epoch  : {best_epoch}/{cfg.total_epochs}')
print(f'  Val  top-1  : {val_top1*100:.2f}%')
print(f'  Test top-1  : {test_top1*100:.2f}%  ← target: >80%')
print(f'  Test top-5  : {test_top5*100:.2f}%')
print(f'  Checkpoint  : {best_ckpt_path}')

Summary saved → models\results\wikiart_tests_1_to_12_summary.csv

=== FINAL RESULTS ===
  Model       : vit_base_patch14_reg4_dinov2.lvd142m
  Best epoch  : 72/85
  Val  top-1  : 78.31%
  Test top-1  : 77.33%  ← target: >80%
  Test top-5  : 96.49%
  Checkpoint  : models\wikiart_test12_vitlarge_448_best.pt


C:\Users\Thijs\AppData\Local\Temp\ipykernel_38316\3987552826.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'saved_at_utc':   datetime.datetime.utcnow().isoformat(),
